# DMIF — Serve Notebook (Flask + ngrok)
This is the LIVE prediction API. Run this whenever you want the web app's real predictions to work (not demo mode).

Restart this notebook -> get a NEW ngrok URL each time -> paste that URL into your Vercel project's `PREDICTION_API_BASE_URL` environment variable.

Uses your confirmed winning models: **BiLSTM** + **MobileNetV2 (candlestick)** + the trained **Fusion layer**.

# Keep-alive

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

<IPython.core.display.Javascript object>

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install and Imports

In [ ]:
!pip install flask flask-cors pyngrok --quiet

import os
import io
import json
import pickle
import threading
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from flask import Flask, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: []


# Configuration

In [ ]:
MODELS_DIR  = '/content/drive/MyDrive/DMIF/models'
LSTM_PATH   = f'{MODELS_DIR}/lstm_bilstm.keras'
CNN_PATH    = f'{MODELS_DIR}/cnn_mobilenet_candlestick.keras'
BUNDLE_PATH = f'{MODELS_DIR}/fusion_bundle.pkl'

CSV_PATH    = '/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv'
LOCAL_CSV   = '/content/master_dataset.csv'

SEQ_LEN     = 30
IMG_SIZE    = 64

# Dataset's actual last trading day — genuine "next day" prediction is for the day after this
DATASET_LAST_TRADING_DATE = '2026-02-24'

LSTM_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Log_Volume',
    'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range',
    'MA_24', 'MA_30', 'MA_500',
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width',
    'Volatility_10', 'SAR', 'SAR_Trend',
    'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786',
    'Volume_MA_10', 'Volume_Ratio'
]

# NGROK AUTH TOKEN — paste your token here (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "your_actual_token_here"

print(f"LSTM   : {LSTM_PATH}")
print(f"CNN    : {CNN_PATH}")
print(f"Bundle : {BUNDLE_PATH}")

LSTM   : /content/drive/MyDrive/DMIF/models/lstm_bilstm.keras
CNN    : /content/drive/MyDrive/DMIF/models/cnn_mobilenet_candlestick.keras
Bundle : /content/drive/MyDrive/DMIF/models/fusion_bundle.pkl


# Load Models, Fusion Bundle, and Dataset

In [ ]:
print("Loading BiLSTM...")
lstm_model = load_model(LSTM_PATH)

print("Loading MobileNetV2 (candlestick)...")
cnn_model = load_model(CNN_PATH)

print("Loading fusion bundle...")
with open(BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

fusion_clf     = bundle['fusion_clf']
best_threshold = bundle['best_threshold']
weights_norm   = bundle['weights_norm']   # [lstm_weight, cnn_weight], normalized

print("Copying master dataset to local disk...")
import shutil
shutil.copy(CSV_PATH, LOCAL_CSV)

df = pd.read_csv(LOCAL_CSV, parse_dates=['Date'])
df = df.sort_values(['Company_Code', 'Date']).reset_index(drop=True)

companies = sorted(df['Company_Code'].unique().tolist())

print(f"All loaded. {len(companies)} companies available.")
print(f"Fusion threshold : {best_threshold:.2f}")
print(f"Fusion weights   : LSTM={weights_norm[0]:.3f}  CNN={weights_norm[1]:.3f}")

Loading BiLSTM...
Loading MobileNetV2 (candlestick)...
Loading fusion bundle...
Copying master dataset to local disk...
All loaded. 80 companies available.
Fusion threshold : 0.35
Fusion weights   : LSTM=0.591  CNN=0.409


# Pre-fit Scalers Per Company
Fit once at startup (not per-request) so predictions are fast.

In [ ]:
scalers = {}

for company in companies:
    company_df = df[df['Company_Code'] == company].copy()
    company_df = company_df.sort_values('Date').reset_index(drop=True)
    company_df = company_df.dropna(subset=LSTM_FEATURES + ['Target'])

    if len(company_df) < SEQ_LEN + 1:
        continue

    train_end = int(len(company_df) * 0.7)
    scaler = MinMaxScaler()
    scaler.fit(company_df[LSTM_FEATURES].iloc[:train_end])
    scalers[company] = scaler

print(f"Scalers fitted for {len(scalers)} companies")

Scalers fitted for 80 companies


# Prediction Function
Mirrors the exact JSON shape the Next.js `/api/predict` route expects: company, direction, up_pct, down_pct, confidence, lstm_pct, cnn_pct, lstm_weight, cnn_weight, as_of.

In [ ]:
def generate_candlestick_image(window_df, img_size=IMG_SIZE):
    """Renders a candlestick chart matching the training data's visual style."""
    fig = plt.figure(figsize=(img_size/100, img_size/100), dpi=100)
    fig.patch.set_facecolor("#0d1117")
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.04)
    ax_p = fig.add_subplot(gs[0])
    ax_v = fig.add_subplot(gs[1])
    for ax in [ax_p, ax_v]:
        ax.set_facecolor("#0d1117")
        ax.tick_params(colors="none", labelsize=0, length=0)
        for s in ax.spines.values():
            s.set_visible(False)

    op = window_df["Open"].values
    hi = window_df["High"].values
    lo = window_df["Low"].values
    cl = window_df["Close"].values

    for i in range(len(window_df)):
        c = "#26a69a" if cl[i] >= op[i] else "#ef5350"
        ax_p.plot([i, i], [lo[i], hi[i]], color=c, linewidth=0.8)
        bh = max(abs(cl[i] - op[i]), max(cl[i], 1) * 0.002)
        ax_p.add_patch(mpatches.Rectangle(
            (i - 0.38, min(op[i], cl[i])), 0.76, bh,
            facecolor=c, edgecolor="none"
        ))

    if "Volume" in window_df.columns:
        vl = window_df["Volume"].values.astype(float)
    elif "Share_Volume" in window_df.columns:
        vl = window_df["Share_Volume"].values.astype(float)
    else:
        vl = np.zeros(len(window_df))
    vc = ["#26a69a" if cl[i] >= op[i] else "#ef5350" for i in range(len(window_df))]
    ax_v.bar(range(len(window_df)), vl, color=vc, width=0.76, alpha=0.65)

    ax_p.set_xlim(-0.5, len(window_df) - 0.5)
    ax_v.set_xlim(-0.5, len(window_df) - 0.5)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, facecolor="#0d1117")
    plt.close(fig)
    buf.seek(0)

    img_arr = np.array(
        Image.open(buf).convert("RGB").resize((img_size, img_size)),
        dtype=np.float32
    ) / 255.0
    return img_arr


def predict_company(ticker, target_date=None):
    ticker = ticker.upper()
    if ticker not in scalers:
        return {"error": f"{ticker} not in trained companies"}, 404

    company_df = df[df["Company_Code"] == ticker].sort_values("Date").reset_index(drop=True)
    window = company_df.iloc[-SEQ_LEN:].copy()

    if len(window) < SEQ_LEN:
        return {"error": f"Not enough history for {ticker}"}, 400

    # ── LSTM branch ──────────────────────────────────────────────
    scaler = scalers[ticker]
    feat_data = window[LSTM_FEATURES].ffill().fillna(0).values
    scaled = scaler.transform(feat_data)
    lstm_prob = float(lstm_model.predict(scaled[np.newaxis], verbose=0)[0][0])

    # ── CNN branch ───────────────────────────────────────────────
    img_arr = generate_candlestick_image(window, IMG_SIZE)
    cnn_prob = float(cnn_model.predict(img_arr[np.newaxis], verbose=0)[0][0])

    # ── Fusion ───────────────────────────────────────────────────
    meta_in = np.array([[lstm_prob, cnn_prob]])
    up_prob = float(fusion_clf.predict_proba(meta_in)[0][1])
    direction = "UP" if up_prob >= best_threshold else "DOWN"

    last_known_date = company_df["Date"].iloc[-1]
    next_trading_date = (last_known_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    result = {
        "company"      : ticker,
        "direction"    : direction,
        "up_pct"       : round(up_prob * 100, 1),
        "down_pct"     : round((1 - up_prob) * 100, 1),
        "confidence"   : round(abs(up_prob - best_threshold) / max(best_threshold, 1 - best_threshold) * 100, 1),
        "lstm_pct"     : round(lstm_prob * 100, 1),
        "cnn_pct"      : round(cnn_prob * 100, 1),
        "lstm_weight"  : round(float(weights_norm[0]) * 100),
        "cnn_weight"   : round(float(weights_norm[1]) * 100),
        "as_of"        : next_trading_date,
        "threshold_used": best_threshold,
    }

    # ── Multi-step forecast for dates beyond next trading day ─────
    if target_date:
        target_ts = pd.Timestamp(target_date)
        next_ts    = pd.Timestamp(next_trading_date)
        if target_ts > next_ts:
            steps = (target_ts - next_ts).days
            steps = max(1, min(steps, 60))  # cap runaway iteration
            decayed_confidence = result["confidence"] * (0.9 ** steps)
            result["confidence"]     = round(decayed_confidence, 1)
            result["is_multi_step"]  = True
            result["forecast_steps"] = steps
            result["as_of"]          = target_date
        else:
            result["is_multi_step"] = False

    return result, 200

print("Prediction function ready")

Prediction function ready


# Test the Prediction Function Locally
(before exposing it via Flask — quick sanity check)

In [ ]:
test_ticker = companies[0]
result, status = predict_company(test_ticker)
print(f"Test prediction for {test_ticker} (status {status}):")
print(json.dumps(result, indent=2))

Test prediction for AAF.N (status 200):
{
  "company": "AAF.N",
  "direction": "DOWN",
  "up_pct": 30.5,
  "down_pct": 69.5,
  "confidence": 6.9,
  "lstm_pct": 49.7,
  "cnn_pct": 43.9,
  "lstm_weight": 59,
  "cnn_weight": 41,
  "as_of": "2026-02-24",
  "threshold_used": 0.35000000000000003
}


# Flask API

In [ ]:
app = Flask(__name__)
CORS(app)

@app.route("/companies")
def get_companies():
    return jsonify({"companies": companies})

@app.route("/predict/<ticker>")
def predict(ticker):
    target_date = request.args.get("targetDate", None)
    result, status = predict_company(ticker, target_date)
    return jsonify(result), status

@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "companies": len(companies),
        "models": {
            "lstm": "bilstm",
            "cnn": "mobilenet_candlestick",
            "fusion_threshold": best_threshold
        },
        "dataset_last_trading_date": DATASET_LAST_TRADING_DATE
    })

print("Flask app configured — routes: /companies, /predict/<ticker>, /health")

Flask app configured — routes: /companies, /predict/<ticker>, /health


# SHAP Explainability Setup

In [ ]:
import shap
import numpy as np

print("Building SHAP background summary (one-time setup)...")

np.random.seed(42)
bg_companies = np.random.choice(companies, size=min(15, len(companies)), replace=False)
bg_sequences = []

for co in bg_companies:
    co_df = df[df["Company_Code"] == co].sort_values("Date").reset_index(drop=True)
    if len(co_df) < SEQ_LEN:
        continue
    window = co_df.iloc[-SEQ_LEN:]
    scaler = scalers.get(co)
    if scaler is None:
        continue
    feat_data = window[LSTM_FEATURES].ffill().fillna(0).values
    scaled = scaler.transform(feat_data)
    bg_sequences.append(scaled)

X_background = np.array(bg_sequences, dtype=np.float32)
X_background_flat = X_background.reshape(X_background.shape[0], SEQ_LEN * len(LSTM_FEATURES))

def shap_predict_fn(x_flat):
    x_reshaped = x_flat.reshape(-1, SEQ_LEN, len(LSTM_FEATURES))
    return lstm_model.predict(x_reshaped, verbose=0).flatten()

background_summary = shap.kmeans(X_background_flat, 8)
shap_explainer = shap.KernelExplainer(shap_predict_fn, background_summary)

print(f"SHAP explainer ready (background: {len(X_background)} sequences)")


def explain_prediction(ticker):
    ticker = ticker.upper()
    if ticker not in scalers:
        return {"error": f"{ticker} not in trained companies"}, 404

    company_df = df[df["Company_Code"] == ticker].sort_values("Date").reset_index(drop=True)
    window = company_df.iloc[-SEQ_LEN:].copy()

    scaler = scalers[ticker]
    feat_data = window[LSTM_FEATURES].ffill().fillna(0).values
    scaled = scaler.transform(feat_data)
    x_flat = scaled.reshape(1, SEQ_LEN * len(LSTM_FEATURES))

    shap_vals_flat = shap_explainer.shap_values(x_flat, nsamples=30, silent=True)
    if isinstance(shap_vals_flat, list):
        shap_vals_flat = shap_vals_flat[0]

    shap_vals = np.array(shap_vals_flat).reshape(SEQ_LEN, len(LSTM_FEATURES))
    feature_importance = shap_vals.mean(axis=0)

    ranked = sorted(
        zip(LSTM_FEATURES, feature_importance),
        key=lambda x: abs(x[1]),
        reverse=True
    )

    top_features = [
        {
            "feature": name,
            "contribution": round(float(value), 6),
            "direction": "UP" if value > 0 else "DOWN"
        }
        for name, value in ranked[:5]
    ]

    return {
        "company": ticker,
        "top_features": top_features,
        "note": "Approximate explanation using a reduced sample size for responsiveness."
    }, 200


@app.route("/explain/<ticker>")
def explain(ticker):
    result, status = explain_prediction(ticker)
    return jsonify(result), status

print("SHAP endpoint ready: /explain/<ticker>")

Building SHAP background summary (one-time setup)...
SHAP explainer ready (background: 15 sequences)
SHAP endpoint ready: /explain/<ticker>


# Start ngrok Tunnel + Run Flask
This cell blocks/keeps running — that's expected, it's serving the API. Copy the printed URL into your Vercel project's `PREDICTION_API_BASE_URL` environment variable.

In [ ]:
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("WARNING: No ngrok auth token set — tunnel may fail to start.")
    print("Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken")

url = ngrok.connect(5000).public_url

print("=" * 55)
print(f"API URL: {url}")
print()
print("COPY THIS URL into your Vercel project settings as:")
print(f'  PREDICTION_API_BASE_URL = {url}')
print("=" * 55)

threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000, use_reloader=False),
    daemon=True
).start()

print("API running. Keep this cell alive — do not restart the runtime.")

API URL: https://recycling-suspect-transfer.ngrok-free.dev

COPY THIS URL into your Vercel project settings as:
  PREDICTION_API_BASE_URL = https://recycling-suspect-transfer.ngrok-free.dev
API running. Keep this cell alive — do not restart the runtime.
